## Limpieza detallada
#### En este notebook podrá ejecutar el proceso de limpieza paso a paso, observando los diagnósticos de suciedad de la data.

In [ ]:
import pandas as pd
import re
import warnings
import os
warnings.filterwarnings('ignore')

In [ ]:
# Carga de la data
ruta_externa = os.path.abspath("../Data")
ruta_archivo = pd.read_csv(f'{ruta_externa}/Data_Banreservas_Original.csv')

try:
  df = pd.read_csv(ruta_archivo, sep=',', encoding='utf-8')
except:
  df = pd.read_csv(ruta_archivo, sep=';', encoding='utf-8')

print("--- TODAS LAS COLUMNAS DEL DATASET ---")
print(df.columns.tolist())

print("\n--- PRIMERAS 3 FILAS (HEAD) ---")
display(df.head(3))



--- TODAS LAS COLUMNAS DEL DATASET ---
['Date', 'Time', 'Document ID', 'URL', 'Input Name', 'Keywords', 'Information Type', 'Source Type', 'Source Name', 'Source Domain', 'Content Type', 'Author Name', 'Author Handle', 'Title', 'Opening Text', 'Hit Sentence', 'Image', 'Hashtags', 'Links', 'Country', 'Region', 'State', 'City', 'Language', 'Sentiment', 'Keyphrases', 'Reach', 'Global Reach', 'National Reach', 'Local Reach', 'Episode Reach', 'AVE', 'EMV', 'Social Echo', 'Editorial Echo', 'Engagement', 'Shares', 'Quotes', 'Likes', 'Replies', 'Reposts', 'Comments', 'Reactions', 'Views', 'Estimated Views', 'Document Tags', 'Custom Categories', 'Custom Fields', 'Brand Sentiment']

--- PRIMERAS 3 FILAS (HEAD) ---


,Date,Time,Document ID,URL,Input Name,Keywords,Information Type,Source Type,Source Name,Source Domain,...,Replies,Reposts,Comments,Reactions,Views,Estimated Views,Document Tags,Custom Categories,Custom Fields,Brand Sentiment
0,30/6/2025,23:58,"""1751342295000_SurW75nfLlvU0m3xEZMNhT1fWjwA""",https://twitter.com/unstoppable8672/statuses/1...,BanReservas OR Banco de Reservas,Banreservas,social,social network,Twitter,twitter.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown
1,30/6/2025,23:38,"""1751341139000_d46Z8XxSNcEaqgE5cfnSJuVWaHcA""",https://twitter.com/Camilocuryv/statuses/19398...,BanReservas OR Banco de Reservas,Banreservas,social,social network,Twitter,twitter.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown
2,30/6/2025,23:24,"""1751340275000_DS6Eat50RKdbBiQgefq0vlfULecA""",https://twitter.com/feyesperanza03/statuses/19...,BanReservas OR Banco de Reservas,Banreservas,social,social network,Twitter,twitter.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown


In [13]:
# Filtramos solo las variables de interés conceptual
columnas_clave = [
'Document ID', 'Date', 'Time', 'URL', 'Author Name', 'Author Handle',
'Hit Sentence', 'Hashtags', 'Keyphrases',
'Reach', 'Engagement', 'Likes', 'Replies', 'Reposts', 'Views'
]

df_filtered = df[columnas_clave].copy()

print(f"Nos quedamos con {df_filtered.shape[1]} columnas clave.")


Nos quedamos con 15 columnas clave.


In [14]:
# Identificamos columnas para comparar (todas menos Document ID)
columnas_para_comparar = [col for col in df_filtered.columns if col != 'Document ID']

# Contamos cuántos hay antes
cantidad_antes = len(df_filtered)

# Eliminamos los que tengan la misma información en el resto de las columnas
df_filtered = df_filtered.drop_duplicates(subset=columnas_para_comparar, keep='first')

duplicados_eliminados = cantidad_antes - len(df_filtered)

print(f"--- ELIMINACIÓN DE DUPLICADOS ---")
print(f"Se encontraron y eliminaron {duplicados_eliminados} registros duplicados de contenido.")
print(f"Total de registros limpios: {len(df_filtered)}")

--- ELIMINACIÓN DE DUPLICADOS ---
Se encontraron y eliminaron 94 registros duplicados de contenido.
Total de registros limpios: 21445


In [15]:
# Calculamos nulos y porcentajes
nulos_cantidad = df_filtered.isnull().sum()
nulos_porcentaje = (nulos_cantidad / len(df_filtered)) * 100

tabla_nulos = pd.DataFrame({
'Nulos (Cantidad)': nulos_cantidad,
'Porcentaje (%)': nulos_porcentaje.round(2)
})

# Filtramos para ver solo las columnas que están sucias
tabla_nulos = tabla_nulos[tabla_nulos['Nulos (Cantidad)'] > 0].sort_values(by='Porcentaje (%)', ascending=False)

print("--- REPORTE DE VALORES NULOS ---")
display(tabla_nulos)



--- REPORTE DE VALORES NULOS ---


,Nulos (Cantidad),Porcentaje (%)
Replies,19286,89.93
Reposts,19119,89.15
Likes,17120,79.83
Engagement,16195,75.52
Views,16065,74.91
Hashtags,13002,60.63
Keyphrases,10374,48.37
Author Name,1071,4.99
Author Handle,1071,4.99


In [16]:
print("--- IMPUTACIÓN Y TRANSFORMACIÓN ---")

# 1. Tratamiento de valores faltantes (Imputación)
metricas_sociales = ['Reach', 'Engagement', 'Likes', 'Replies', 'Reposts', 'Views']

# Llenamos los nulos numéricos con 0
for col in metricas_sociales:
  df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)

# Eliminamos los comentarios sin autor (estos son no disponibles)
df_filtered = df_filtered.dropna(subset=['Author Handle'])

# Llenamos los nulos categóricos con etiquetas descriptivas
df_filtered['Hashtags'] = df_filtered['Hashtags'].fillna('Sin_Hashtag')
df_filtered['Keyphrases'] = df_filtered['Keyphrases'].fillna('Ninguna')
df_filtered['Hit Sentence'] = df_filtered['Hit Sentence'].fillna('')

# 2. Corrección de formatos de datos
df_filtered['Date'] = pd.to_datetime(df_filtered['Date'], errors='coerce')

# 3. Transformaciones necesarias en las variables (Limpieza NLP)
def procesar_texto_nlp(texto: str) -> str:
  if pd.isna(texto):
    return ""
  texto = str(texto).lower()
  texto = re.sub(r'http\S+|www\.\S+', '', texto)
  texto = re.sub(r'@\w+', '', texto)
  texto = re.sub(r'[^\w\s]', ' ', texto)
  texto = re.sub(r'\s+', ' ', texto).strip()
  return texto

# Aplicamos la función a la nueva columna
df_filtered['Cleaned_Text'] = df_filtered['Hit Sentence'].apply(procesar_texto_nlp)

# Comprobación final
print(f"Cantidad de nulos restantes en el dataset: {df_filtered.isnull().sum().sum()}")

--- IMPUTACIÓN Y TRANSFORMACIÓN ---
Cantidad de nulos restantes en el dataset: 0


In [18]:
# Eliminamos registros de comentarios de @Banreservasrd y @segurosreservas
invalid_users = ['@Banreservas', '@banreservasrd', '@BanreservasRD', '@segurosreservas']

df_filtered = df_filtered[
    (~df['Author Handle'].isin(invalid_users))
]

In [19]:
# Búsqueda de comentarios en blanco o que contengan únicamente elementos descartados, como enlaces
print(f"Cantidad de textos en blanco restantes en el dataset: {(df_filtered['Cleaned_Text'] == '').sum()}")

# Eliminación de comentarios en blanco
df_filtered = df_filtered[df_filtered['Cleaned_Text'] != '']

print(f"Cantidad de textos en blanco restantes en el dataset tras limpieza: {(df_filtered['Cleaned_Text'] == '').sum()}")

Cantidad de textos en blanco restantes en el dataset: 1
Cantidad de textos en blanco restantes en el dataset tras limpieza: 0


In [20]:
# 4. Exportar la data totalmente limpia y lista
ruta_final = 'Data_Banreservas_Cleaned.csv'
df_filtered.to_csv(ruta_final, index=False, encoding='utf-8-sig')